### Transducers for a Moving Standing Wave

**Assumptions:** Plane waves along the axis, equal amplitudes, air at 20°C, and rough attenuation of $1.6\,\text{dB/cm}$ at $1\,\text{MHz}$ scaling with $f^2$. No transducer phase lag (calibrate that out).


Drive settings for two opposing transducers on the jet axis:

$$\text{upstream transducer} \longrightarrow \text{interaction point} \longrightarrow \text{downstream transducer}$$

Two opposing waves of slightly different frequency make a standing wave whose nodes drift toward the slower one:

* **Node spacing:**
  $$\Delta = \frac{c}{f_{\text{up}} + f_{\text{down}}}$$
* **Node (envelope) velocity:**
  $$v = c\,\frac{f_{\text{up}} - f_{\text{down}}}{f_{\text{up}} + f_{\text{down}}}$$

Solving for the transducer frequencies gives:

$$f_{\text{up}} = \frac{c + v}{2\Delta}, \qquad f_{\text{down}} = \frac{c - v}{2\Delta}, \qquad f_{\text{up}} - f_{\text{down}} = \frac{v}{\Delta}$$

where $f_{\text{up}} - f_{\text{down}} = v / \Delta$ is the beat frequency (the rate at which nodes pass a fixed point).

All positions are measured from the nozzle exit, and time is the model's $t$: $t = 0$ is a rising zero crossing of the nozzle drive,

$$h(t) = h_0\bigl(1 + \delta \sin(2\pi f_{\text{nozzle}} t)\bigr)$$

---

### Upper & Lower Critical Velocities (Rayleigh Jetting)

For stable Rayleigh jetting, the Weber number must satisfy both the dripping threshold and the aerodynamic breakup limit:

$$\mathrm{We} = \frac{\rho_l v^2 r}{\gamma} > 4 \qquad \text{and} \qquad \left(\frac{\rho_g}{\rho_l}\right)\mathrm{We} < 1$$

* **Lower critical velocity** (dripping-to-jetting transition):
  $$\frac{\rho_l v^2 r}{\gamma} > 4 \implies v^2 > \frac{4\gamma}{\rho_l r} \implies v > 2\sqrt{\frac{\gamma}{\rho_l r}}$$

* **Upper critical velocity** (aerodynamic breakup limit):
  $$\frac{\rho_g}{\cancel{\rho_l}} \cdot \frac{\cancel{\rho_l} v^2 r}{\gamma} < 1 \implies v < \sqrt{\frac{\gamma}{\rho_g r}}$$

Combining both bounds gives the operating window for jet velocity $v$:

$$2\sqrt{\frac{\gamma}{\rho_l r}} < v < \sqrt{\frac{\gamma}{\rho_g r}}$$

In [9]:
import math
from IPython.display import display, Markdown

# Fluid properties of 40% w/w glycerol-saline (20 °C); water values kept below for reference
gasDensity = 1.204                      # [kg/m^3]    (air at 20 °C)
liquidDensity = 998.2                   # [kg/m^3]    (pure water at 20 °C)
gamma_w = 0.0728                        # [N/m]       (surface tension of water in air at 20 °C)
viscosity_w = 1.002e-3                  # [Pa·s]      (dynamic viscosity of water at 20 °C)
liquidSoundSpeed = 1481.0               # [m/s]       (speed of sound in water at 20 °C)
# gasDensity = 1.204                      # [kg/m^3]    (air at 20 °C)
# liquidDensity = 1098.0                  # [kg/m^3]    (40% w/w glycerol-saline, Van Hoeve et al. 2010, matches main.cpp)
# gamma_w = 0.0679                        # [N/m]       (surface tension of the solution in air)
# viscosity_w = 3.65e-3                   # [Pa·s]      (dynamic viscosity of the solution)
# liquidSoundSpeed = 1580.0               # [m/s]       (speed of sound in ~40% w/w glycerol-water at 20 °C, approximate)

# Air Properties
airDensity = 1.204                      # [kg/m^3]
airSoundSpeed = 343.2                   # [m/s]
attenuationAt1MHz = 1.6                 # [dB/cm], rough (depends on humidity?). Adjust given target frequency.

# Geom
interactionPoint = 5e-3
upstreamTransducerPosition = 0.0        # annular transducer in the nozzle plane
downstreamTransducerPosition = 10e-3    # maybe something like 2x distance from IP?
transducerRadius = 5e-3                 # radiating face radius, for rough power estimates

# Microdroplet Stream
frequency = 26e3                        # [hz]
dropRadius = 120e-6                     # [m]
nozzleRadius = 60e-6                    # [m]

### Compute Critical Velocities for Rayleigh Breakup
critV = lambda nozzleRadius, liquidDensity, gasDensity, gamma_w : [2 * math.sqrt(gamma_w / liquidDensity / nozzleRadius), math.sqrt(gamma_w / gasDensity / nozzleRadius)]
v_min, v_max = critV(nozzleRadius, liquidDensity, gasDensity, gamma_w)

flowRate = frequency * 4/3 * math.pi * (dropRadius)**3
jetVelocity = flowRate / (math.pi * nozzleRadius**2) # [m/s]

# Target Standing wave
pressureAmplitude = 2000             # Pa, peak pressure at an antinode
dropletSpacing = 300e-6               # m
pressureNodesPerDropletSpacing = 1 
patternVelocity = 4.5                  # m/s, node velocity (= drop velocity)
nodePositionAtStart = 72.7e-6           # m, a node is here at t = 0

# Jet Excitations
nozzleFrequency = jetVelocity / (2 * math.sqrt(2) * math.pi * nozzleRadius)   # Hz, vjet / lambda_opt (main.cpp)

### Property table printout
rows = [
    ("Nozzle radius ($r$)", f"{nozzleRadius * 1e6:.1f}", r"$\mu\text{m}$"),
    ("Target drop radius ($R$)", f"{dropRadius * 1e6:.1f}", r"$\mu\text{m}$"),
    ("Drop frequency ($f$)", f"{frequency * 1e-3:.2f}", r"$\text{kHz}$"),
    ("Rayleigh optimal frequency ($f_{\\text{nozzle}}$)", f"{nozzleFrequency * 1e-3:.2f}", r"$\text{kHz}$"),
    ("Liquid density ($\\rho_l$)", f"{liquidDensity:.1f}", r"$\text{kg/m}^3$"),
    ("Gas density ($\\rho_g$)", f"{gasDensity:.3f}", r"$\text{kg/m}^3$"),
    ("Surface tension ($\\gamma$)", f"{gamma_w:.4f}", r"$\text{N/m}$"),
    ("Dynamic viscosity ($\\mu$)", f"{viscosity_w * 1e3:.3f}", r"$\text{mPa}\cdot\text{s}$"),
    ("Lower crit. velocity ($v_{\\min}$)", f"{v_min:.2f}", r"$\text{m/s}$"),
    ("Upper crit. velocity ($v_{\\max}$)", f"{v_max:.2f}", r"$\text{m/s}$"),
    ("Jet velocity ($v$)", f"{jetVelocity:.2f}", r"$\text{m/s}$"),
    ("Flow rate ($\\dot{V}$)", f"{flowRate * 1e6:.4f}", r"$\text{mL/s}$"),
]
md_table = "| Property | Value | Unit |\n| :--- | ---: | :--- |\n" + "\n".join(
    f"| {prop} | {val} | {unit} |" for prop, val, unit in rows
)
display(Markdown(md_table))


| Property | Value | Unit |
| :--- | ---: | :--- |
| Nozzle radius ($r$) | 60.0 | $\mu\text{m}$ |
| Target drop radius ($R$) | 120.0 | $\mu\text{m}$ |
| Drop frequency ($f$) | 26.00 | $\text{kHz}$ |
| Rayleigh optimal frequency ($f_{\text{nozzle}}$) | 31.21 | $\text{kHz}$ |
| Liquid density ($\rho_l$) | 998.2 | $\text{kg/m}^3$ |
| Gas density ($\rho_g$) | 1.204 | $\text{kg/m}^3$ |
| Surface tension ($\gamma$) | 0.0728 | $\text{N/m}$ |
| Dynamic viscosity ($\mu$) | 1.002 | $\text{mPa}\cdot\text{s}$ |
| Lower crit. velocity ($v_{\min}$) | 2.21 | $\text{m/s}$ |
| Upper crit. velocity ($v_{\max}$) | 31.75 | $\text{m/s}$ |
| Jet velocity ($v$) | 16.64 | $\text{m/s}$ |
| Flow rate ($\dot{V}$) | 0.1882 | $\text{mL/s}$ |

### Nozzle Sizing & Optimal Rayleigh Droplet Size

Operate $15\%$ of the way from the lower to the upper critical velocity within the Rayleigh window:

$$v = v_{\min} + 0.15\,(v_{\max} - v_{\min})$$

For an inviscid jet the optimum (fastest-growing) Rayleigh wavelength depends only on the jet radius:

$$\lambda_{\text{opt}} = 2\sqrt{2}\,\pi r \approx 8.886\,r = 4.443\,d$$

For a viscous jet, the optimum wavelength also depends on the Ohnesorge number $\mathrm{Oh} = \mu / \sqrt{\rho_l \gamma r}$, which compares viscous forces to inertia and surface tension. The fastest-growing wavenumber satisfies:

$$k_{\text{opt}}\,r = \frac{1}{\sqrt{2 + 3\sqrt{2}\,\mathrm{Oh}}} \qquad \Longrightarrow \qquad \lambda_{\text{opt}} = \frac{2\pi r}{k_{\text{opt}}\,r} = 2\sqrt{2}\,\pi r\,\sqrt{1 + \tfrac{3}{\sqrt{2}}\,\mathrm{Oh}}$$

The optimal breakup frequency is:

$$f_{\text{opt}} = \frac{v}{\lambda_{\text{opt}}}$$

Volume conservation ($\dot{V} = f_{\text{opt}} \cdot \frac{4}{3}\pi R^3 = \pi r^2 v \implies \frac{4}{3}\pi R^3 = \pi r^2 \lambda_{\text{opt}}$) gives the droplet radius $R$ and diameter $D = 2R$:

$$R = \left(\frac{3}{4} r^2 \lambda_{\text{opt}}\right)^{1/3} = \left(\frac{3\sqrt{2}\,\pi}{2}\right)^{1/3} r \approx 1.881\,r \qquad \Longleftrightarrow \qquad D \approx 1.881\,d$$

**Inverse problem.** Given a target droplet diameter $D$ and optimal frequency $f_{\text{opt}}$, the nozzle diameter and jet velocity follow directly:

$$d = \frac{D}{1.881}, \qquad v = f_{\text{opt}}\,\lambda_{\text{opt}} = 4.443\,f_{\text{opt}}\,d$$

The result is only usable if $v$ falls inside the Rayleigh window $v_{\min}(d) < v < v_{\max}(d)$, which the table below checks.

In [10]:
## Calculate range of nozzle sizes, critical velocities, and optimal Rayleigh droplet size for the glycerol solution
lambdaOptPerRadius = 2 * math.sqrt(2) * math.pi  # lambda_opt / r for an inviscid jet (≈ 8.886)
nozzle_rows = []
for r in [20e-6, 30e-6, 40e-6, 50e-6, 60e-6, 70e-6, 80e-6, 90e-6]:
    d = 2 * r
    vL, vU = critV(r, liquidDensity, gasDensity, gamma_w)
    v_op = vL + 0.15 * (vU - vL)
    lambda_opt = lambdaOptPerRadius * r  # inviscid optimum wavelength, depends on jet radius only
    f_opt = v_op / lambda_opt
    flow_rate = math.pi * (r ** 2) * v_op
    drop_volume = flow_rate / f_opt         # = pi * r^2 * lambda_opt
    D_opt = 2 * (3 * drop_volume / (4 * math.pi)) ** (1 / 3)
    nozzle_rows.append((
        f"{d * 1e6:.0f}",
        f"{vL:.2f}",
        f"{vU:.2f}",
        f"{v_op:.2f}",
        f"{f_opt * 1e-3:.2f}",
        f"{D_opt * 1e6:.1f}"
    ))

md_nozzle_table = (
    "| Nozzle Diameter $d$ ($\\mu\\text{m}$) | $v_{\\min}$ ($\\text{m/s}$) | $v_{\\max}$ ($\\text{m/s}$) | $v_{15\\%}$ ($\\text{m/s}$) | Optimal Freq. $f_{\\text{opt}}$ ($\\text{kHz}$) | Droplet Diameter $D$ ($\\mu\\text{m}$) |\n"
    "| ---: | ---: | ---: | ---: | ---: | ---: |\n"
    + "\n".join(
        f"| {d_um} | {vL} | {vU} | {v_op} | {f_khz} | {D_um} |"
        for d_um, vL, vU, v_op, f_khz, D_um in nozzle_rows
    )
)
display(Markdown(md_nozzle_table))


| Nozzle Diameter $d$ ($\mu\text{m}$) | $v_{\min}$ ($\text{m/s}$) | $v_{\max}$ ($\text{m/s}$) | $v_{15\%}$ ($\text{m/s}$) | Optimal Freq. $f_{\text{opt}}$ ($\text{kHz}$) | Droplet Diameter $D$ ($\mu\text{m}$) |
| ---: | ---: | ---: | ---: | ---: | ---: |
| 40 | 3.82 | 54.98 | 11.49 | 64.68 | 75.3 |
| 60 | 3.12 | 44.89 | 9.38 | 35.21 | 112.9 |
| 80 | 2.70 | 38.88 | 8.13 | 22.87 | 150.5 |
| 100 | 2.42 | 34.78 | 7.27 | 16.36 | 188.2 |
| 120 | 2.21 | 31.75 | 6.64 | 12.45 | 225.8 |
| 140 | 2.04 | 29.39 | 6.14 | 9.88 | 263.5 |
| 160 | 1.91 | 27.49 | 5.75 | 8.08 | 301.1 |
| 180 | 1.80 | 25.92 | 5.42 | 6.78 | 338.7 |

In [11]:
## Inverse, nozzle diameter and jet velocity from optimal frequency and droplet diameter
dropToNozzle = (3 * lambdaOptPerRadius / 4) ** (1 / 3)   # D / d = R / r ≈ 1.881

targets = [(10, 130), (20, 130), (26, 130), (30, 130), (40, 130), (50, 130)]   # (f_opt [kHz], D [µm])
inverse_rows = []
for f_khz, D_um in targets:
    D = D_um * 1e-6
    d = D / dropToNozzle
    lambda_opt = lambdaOptPerRadius * d / 2              # 2*sqrt(2)*pi*r
    v = f_khz * 1e3 * lambda_opt
    vL, vU = critV(d / 2, liquidDensity, gasDensity, gamma_w)
    position = 100 * (v - vL) / (vU - vL)                # % of the way through the Rayleigh window
    status = "yes" if vL < v < vU else "**no**"
    inverse_rows.append((f_khz, D_um, f"{d * 1e6:.1f}", f"{v:.2f}", f"{vL:.2f}", f"{vU:.2f}", f"{position:.0f}%", status))

md_inverse_table = (
    "| Optimal Freq. $f_{\\text{opt}}$ ($\\text{kHz}$) | Droplet Diameter $D$ ($\\mu\\text{m}$) | Nozzle Diameter $d$ ($\\mu\\text{m}$) | Jet Velocity $v$ ($\\text{m/s}$) | $v_{\\min}$ | $v_{\\max}$ | Window Position | In Window? |\n"
    "| ---: | ---: | ---: | ---: | ---: | ---: | ---: | :---: |\n"
    + "\n".join(" | ".join(["", *map(str, row), ""]).strip() for row in inverse_rows)
)
display(Markdown(md_inverse_table))

| Optimal Freq. $f_{\text{opt}}$ ($\text{kHz}$) | Droplet Diameter $D$ ($\mu\text{m}$) | Nozzle Diameter $d$ ($\mu\text{m}$) | Jet Velocity $v$ ($\text{m/s}$) | $v_{\min}$ | $v_{\max}$ | Window Position | In Window? |
| ---: | ---: | ---: | ---: | ---: | ---: | ---: | :---: |
| 10 | 130 | 69.1 | 3.07 | 2.91 | 41.84 | 0% | yes |
| 20 | 130 | 69.1 | 6.14 | 2.91 | 41.84 | 8% | yes |
| 26 | 130 | 69.1 | 7.98 | 2.91 | 41.84 | 13% | yes |
| 30 | 130 | 69.1 | 9.21 | 2.91 | 41.84 | 16% | yes |
| 40 | 130 | 69.1 | 12.28 | 2.91 | 41.84 | 24% | yes |
| 50 | 130 | 69.1 | 15.35 | 2.91 | 41.84 | 32% | yes |

In [18]:
## Viscosity sweep: nozzle diameter, jet velocity and critical velocities at fixed rep rate and droplet diameter
# Viscosity enters through the Ohnesorge number, which shifts the fastest-growing mode (Weber):
#   k_opt * r = 1 / sqrt(2 + 3*sqrt(2)*Oh),   Oh = mu / sqrt(rho * gamma * r)   (Oh -> 0 recovers 2*sqrt(2)*pi*r)
repRate = 35e3          # [Hz]
dropDiameter = 130e-6   # [m]

def optimalWavenumber(r, mu):
    return 1 / math.sqrt(2 + 3 * math.sqrt(2) * mu / math.sqrt(liquidDensity * gamma_w * r))

def nozzleRadiusFor(D, mu):
    # D = 2 r (3*pi / (2 k_opt r))^(1/3); k_opt r depends weakly on r, so fixed-point iteration converges quickly
    r = D / 2 / dropToNozzle
    for _ in range(100):
        r = D / 2 / (3 * math.pi / (2 * optimalWavenumber(r, mu))) ** (1 / 3)
    return r

viscosities_mPas = [1.0, 2.0, 3.65, 5.0, 8.0, 12.0, 20.0]
viscosity_rows = []
for mu_mPas in viscosities_mPas:
    mu = mu_mPas * 1e-3
    r = nozzleRadiusFor(dropDiameter, mu)
    Oh = mu / math.sqrt(liquidDensity * gamma_w * r)
    lambda_opt = 2 * math.pi * r / optimalWavenumber(r, mu)
    v = repRate * lambda_opt
    vL, vU = critV(r, liquidDensity, gasDensity, gamma_w)
    position = 100 * (v - vL) / (vU - vL)                # % of the way through the Rayleigh window
    status = "yes" if vL < v < vU else "**no**"
    viscosity_rows.append((f"{mu_mPas:g}", f"{Oh:.3f}", f"{2 * r * 1e6:.1f}", f"{lambda_opt * 1e6:.1f}", f"{v:.2f}", f"{vL:.2f}", f"{vU:.2f}", f"{position:.0f}%", status))

md_viscosity_table = (
    f"Rep rate $f = {repRate * 1e-3:.0f}\\,\\text{{kHz}}$, droplet diameter $D = {dropDiameter * 1e6:.0f}\\,\\mu\\text{{m}}$\n\n"
    "| Viscosity $\\mu$ ($\\text{mPa}\\cdot\\text{s}$) | Ohnesorge $\\mathrm{Oh}$ | Nozzle Diameter $d$ ($\\mu\\text{m}$) | $\\lambda_{\\text{opt}}$ ($\\mu\\text{m}$) | Jet Velocity $v$ ($\\text{m/s}$) | $v_{\\min}$ ($\\text{m/s}$) | $v_{\\max}$ ($\\text{m/s}$) | Window Position | In Window? |\n"
    "| ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | :---: |\n"
    + "\n".join("| " + " | ".join(row) + " |" for row in viscosity_rows)
)
display(Markdown(md_viscosity_table))

Rep rate $f = 35\,\text{kHz}$, droplet diameter $D = 130\,\mu\text{m}$

| Viscosity $\mu$ ($\text{mPa}\cdot\text{s}$) | Ohnesorge $\mathrm{Oh}$ | Nozzle Diameter $d$ ($\mu\text{m}$) | $\lambda_{\text{opt}}$ ($\mu\text{m}$) | Jet Velocity $v$ ($\text{m/s}$) | $v_{\min}$ ($\text{m/s}$) | $v_{\max}$ ($\text{m/s}$) | Window Position | In Window? |
| ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | :---: |
| 1 | 0.020 | 68.6 | 311.2 | 10.89 | 2.92 | 41.99 | 20% | yes |
| 2 | 0.040 | 68.1 | 315.4 | 11.04 | 2.93 | 42.13 | 21% | yes |
| 3.65 | 0.074 | 67.4 | 322.2 | 11.28 | 2.94 | 42.35 | 21% | yes |
| 5 | 0.101 | 66.9 | 327.5 | 11.46 | 2.95 | 42.52 | 22% | yes |
| 8 | 0.164 | 65.7 | 339.0 | 11.86 | 2.98 | 42.89 | 22% | yes |
| 12 | 0.248 | 64.4 | 353.4 | 12.37 | 3.01 | 43.34 | 23% | yes |
| 20 | 0.421 | 62.1 | 379.7 | 13.29 | 3.06 | 44.13 | 25% | yes |

In [13]:
# Frequencies
nodeSpacing = dropletSpacing / pressureNodesPerDropletSpacing
frequencySum = airSoundSpeed / nodeSpacing
upstreamFrequency = 0.5 * (airSoundSpeed + patternVelocity) / nodeSpacing
downstreamFrequency = 0.5 * (airSoundSpeed - patternVelocity) / nodeSpacing
beatFrequency = upstreamFrequency - downstreamFrequency          # = patternVelocity / nodeSpacing

print(f"node spacing           {nodeSpacing * 1e6:10.2f} um")
print(f"upstream transducer    {upstreamFrequency:12.1f} Hz")
print(f"downstream transducer  {downstreamFrequency:12.1f} Hz")
print(f"difference             {beatFrequency:12.1f} Hz  (nozzle drive {nozzleFrequency:.1f} Hz)")
dropRate = patternVelocity / dropletSpacing
print(f"drops pass a point at  {dropRate:12.1f} Hz, {100 * (dropRate / nozzleFrequency - 1):+.2f}% from the nozzle frequency")

node spacing               300.00 um
upstream transducer        579500.0 Hz
downstream transducer      564500.0 Hz
difference                  15000.0 Hz  (nozzle drive 31211.0 Hz)
drops pass a point at       15000.0 Hz, -51.94% from the nozzle frequency


drops come off at the nozzle frequency, so set `f_up − f_down = pressureNodesPerDropletSpacing × f_nozzle` exactly (mix or synthesize both from the nozzle clock) and let the node spacing follow. A mismatch of x % slides the nodes off the drops by x % of a spacing every
drop period.

### Phases

Drive signals: upstream $\cos(2\pi f_{\text{up}} t + \text{phaseUp})$, downstream $\cos(2\pi f_{\text{down}} t + 0)$.

With travel delays, the waves at position $z$ are

$$p_{\text{up}} = A\cos(k_{\text{up}} z - \omega_{\text{up}} t + \varphi_{\text{up}}), \qquad p_{\text{down}} = A\cos(k_{\text{down}} z + \omega_{\text{down}} t + \varphi_{\text{down}})$$

where

$$\varphi_{\text{up}} = -k_{\text{up}} z_{\text{up}} - \text{phaseUp}, \qquad \varphi_{\text{down}} = -k_{\text{down}} z_{\text{down}}$$

The nodes sit where

$$\bar{k} z - \pi (f_{\text{up}} - f_{\text{down}})\, t + \frac{\varphi_{\text{up}} + \varphi_{\text{down}}}{2} = \frac{\pi}{2} \pmod{\pi}, \qquad \bar{k} = \frac{k_{\text{up}} + k_{\text{down}}}{2}$$

A node at `nodePositionAtStart` $= z_0$ when $t = 0$ therefore requires

$$\varphi_{\text{up}} + \varphi_{\text{down}} = \pi - 2\bar{k} z_0 \pmod{2\pi}$$

In [14]:
upstreamWavenumber = 2 * math.pi * upstreamFrequency / airSoundSpeed
downstreamWavenumber = 2 * math.pi * downstreamFrequency / airSoundSpeed
meanWavenumber = 0.5 * (upstreamWavenumber + downstreamWavenumber)

downstreamPhase = 0.0
phiDown = -downstreamWavenumber * downstreamTransducerPosition + downstreamPhase
phiUp = math.pi - 2 * meanWavenumber * nodePositionAtStart - phiDown
upstreamPhase = (-upstreamWavenumber * upstreamTransducerPosition - phiUp) % (2 * math.pi)

print(f"upstream drive phase    {math.degrees(upstreamPhase):7.2f} deg   (at t = 0)")
print(f"downstream drive phase  {math.degrees(downstreamPhase):7.2f} deg")

upstream drive phase     105.91 deg   (at t = 0)
downstream drive phase     0.00 deg


In [15]:
# Attenuation
# Amplitude
def attenuationDecibels(frequency, distance):
    return attenuationAt1MHz * (frequency / 1e6) ** 2 * distance * 100

def soundPressureLevel(peakPressure):
    return 20 * math.log10(peakPressure / math.sqrt(2) / 20e-6)

waveAmplitudeAtInteraction = pressureAmplitude / 2
area = math.pi * transducerRadius**2
for name, frequency, position in [("upstream", upstreamFrequency, upstreamTransducerPosition),("downstream", downstreamFrequency, downstreamTransducerPosition)]:
    distance = abs(interactionPoint - position)
    loss = attenuationDecibels(frequency, distance)
    facePressure = waveAmplitudeAtInteraction * 10 ** (loss / 20)
    intensity = facePressure**2 / (2 * airDensity * airSoundSpeed)
    print(
    f"{name:10s}: {distance * 1e3:.1f} mm to the interaction point, {loss:.2f} dB loss -> "
    f"{facePressure:.0f} Pa peak at the face ({soundPressureLevel(facePressure):.1f} dB SPL), "
    f"{intensity / 1e4:.3f} W/cm^2, {intensity * area:.2f} W acoustic"
    )

upstream  : 5.0 mm to the interaction point, 0.27 dB loss -> 1031 Pa peak at the face (151.2 dB SPL), 0.129 W/cm^2, 0.10 W acoustic
downstream: 5.0 mm to the interaction point, 0.25 dB loss -> 1030 Pa peak at the face (151.2 dB SPL), 0.128 W/cm^2, 0.10 W acoustic


In [16]:
# checks
wavelength = airSoundSpeed / (0.5 * frequencySum)
nearField = transducerRadius**2 / wavelength
print(f"acoustic wavelength {wavelength * 1e6:.1f} um; near-field length a^2/λ = {nearField * 1e3:.0f} mm")

acoustic wavelength 600.0 um; near-field length a^2/λ = 42 mm


In [17]:
# check with rebuild
# signal of each wave at (z, t), including travel delay; the node is where |sum| is smallest.
def envelope(z, t):
    up = (2 * math.pi * upstreamFrequency * (t - (z - upstreamTransducerPosition) / airSoundSpeed) + upstreamPhase)
    down = (2 * math.pi * downstreamFrequency * (t - (downstreamTransducerPosition - z) / airSoundSpeed)
            + downstreamPhase)
    return abs(complex(math.cos(up), math.sin(up)) + complex(math.cos(down), math.sin(down)))

for t in [0.0, 10e-6, 25e-6]:
    expected = nodePositionAtStart + patternVelocity * t
    samples = [expected + (i - 500) * nodeSpacing / 1000 for i in range(1001)]  # ± half a spacing
    found = min(samples, key=lambda z: envelope(z, t))
    print(f"t = {t * 1e6:4.0f} us: node expected at {expected * 1e6:8.2f} um, found at {found * 1e6:8.2f} um")

t =    0 us: node expected at    72.70 um, found at    72.70 um
t =   10 us: node expected at   117.70 um, found at   117.70 um
t =   25 us: node expected at   185.20 um, found at   185.20 um
